# Лабораторная работа: Предобработка данных и Машинное обучение

## Цели работы
1. **Очистка данных (Pandas)**: удаление пустых значений (NaN) и исправление типов данных (конвертация текста в числа).
2. **Удаление выбросов (Статистика)**: совместное применение методов **IQR** (межквартильный размах) и **Z-score** (Z-оценка) для фильтрации аномалий аппаратуры.
3. **Машинное обучение (Scikit-Learn)**: применение алгоритмов **Linear Regression** и **Random Forest** для аппроксимации сложной физической модели.
4. **Анализ метрик**: оценка качества моделей с помощью метрик **MAE, MSE, $R^2$**.

---

## 1. Теоретический блок: Электромагнитное экранирование

Мы исследуем эффективность экранирования (SE) однослойных **металлических экранов**.
Точный расчёт SE выполняется аналитически с помощью **Метода матриц передачи (Transfer Matrix Method)**. Этот метод использует гиперболические функции от комплексных аргументов.

Наша задача в машинном обучении — проверить, смогут ли базовые алгоритмы *аппроксимировать* эту сложную физическую функцию на основе трех параметров:
1. **Частота поля ($f$)**: от 1 МГц до 1 ГГц.
2. **Магнитная проницаемость ($\mu_r$)**: от 1 (медь, алюминий) до 100 (сталь, никель).
3. **Удельная электропроводность ($\sigma$)**: от 1 до 100 МСм/м (МегаСименс на метр).

*Толщина экрана строго зафиксирована и равна 1 мкм ($10^{-6}$ м). Диэлектрическая проницаемость $\varepsilon_r = 1$, так как для металлов она не влияет на результат.*

---

## 2. Теория Data Science: Очистка данных и Метрики

### 2.1. Методы поиска выбросов
Данные с реальных датчиков часто содержат аномалии (выбросы). Их нужно удалять перед обучением модели.

1. **Метод межквартильного размаха (IQR)**:
   Отлично подходит для несимметричных распределений (например, для самой величины SE).
   - Вычисляем 25-й перцентиль ($Q1$) и 75-й перцентиль ($Q3$).
   - Рассчитываем размах: $IQR = Q3 - Q1$.
   - Допустимые границы данных: от $[Q1 - 1.5 \times IQR]$ до $[Q3 + 1.5 \times IQR]$.

2. **Метод Z-оценки (Z-score)**:
   Используется для признаков с нормальным или равномерным распределением. Показывает, на сколько стандартных отклонений значение отличается от среднего. Если $|Z| > 3$, значение считается аномалией (правило трёх сигм).

### 2.2. Метрики качества машинного обучения
- **MAE (Mean Absolute Error)**: Средняя абсолютная ошибка. Показывает, на сколько децибел (дБ) в среднем ошибается модель. Метрика легко интерпретируется.
- **MSE (Mean Squared Error)**: Среднеквадратичная ошибка. Сильно "штрафует" модель за крупные промахи, возводя ошибку в квадрат.
- **$R^2$ (Коэффициент детерминации)**: Показывает, насколько хорошо модель описывает данные.
  - $R^2 = 1.0$ — идеальное предсказание.
  - $R^2 \approx 0$ — модель предсказывает случайный шум или константу.

---
## 3. Обучающие примеры (НЕ РЕДАКТИРОВАТЬ)

В этом разделе собраны примеры использования библиотек **Pandas**, **Scikit-Learn** и **Matplotlib**, которые понадобятся вам для выполнения практических заданий. Просто запустите ячейки и изучите результаты.

### 3.1. Пример: Очистка данных от текстового мусора (Pandas)
Часто приборы вместо чисел записывают текст (например, `"сбой"`). Из-за этого весь столбец в Pandas становится текстовым (`object`), и математика перестает работать.

In [ ]:
import pandas as pd
import numpy as np

# === ПРИМЕР 1: Очистка типов данных и удаление NaN ===

# Создаем пример "грязного" датафрейма
df_example = pd.DataFrame({
    'ID_измерения': [1, 2, 3, 4, 5],
    'Проводимость': [50.5, 'ошибка', 42.1, np.nan, 'N/A']
})

print("=== ДО ОЧИСТКИ ===")
print(df_example)
print("\nТипы данных:")
print(df_example.dtypes)

# 1. Принудительно переводим столбец в числа.
# Параметр errors='coerce' превратит весь нечитаемый текст в NaN (Not a Number)
df_example['Проводимость'] = pd.to_numeric(df_example['Проводимость'], errors='coerce')

# 2. Удаляем все строки, в которых есть NaN
df_clean_example = df_example.dropna().reset_index(drop=True)

print("\n=== ПОСЛЕ ОЧИСТКИ ===")
print(df_clean_example)
print("\nТипы данных:")
print(df_clean_example.dtypes)

### 3.2. Пример: Фильтрация данных с помощью булевых масок (Pandas)
Булевы маски позволяют отбирать строки по определенным условиям (например, при удалении выбросов).

In [ ]:
# === ПРИМЕР 2: Булевы маски и фильтрация ===

df_mask_example = pd.DataFrame({
    'Материал': ['Медь', 'Алюминий', 'Сталь', 'Никель', 'Свинец'],
    'SE_дБ': [65, 55, 120, 80, 15] # 120 и 15 - явные аномалии для нашего примера
})

# Задача: оставить только те экраны, где SE находится в пределах от 40 до 100 дБ

# 1. Создаем маску (условие)
# ВАЖНО: Каждое условие берется в скобки (...), между ними ставится амперсанд & (логическое И)
mask = (df_mask_example['SE_дБ'] >= 40) & (df_mask_example['SE_дБ'] <= 100)

print("=== Созданная булева маска ===")
print(mask)

# 2. Применяем маску к датафрейму
df_filtered_example = df_mask_example[mask].reset_index(drop=True)

print("\n=== Отфильтрованный датафрейм ===")
print(df_filtered_example)

### 3.3. Пример: Разделение выборки и обучение Линейной регрессии (Scikit-Learn)
В машинном обучении мы всегда делим данные на **обучающую** (Train) и **тестовую** (Test) выборки, чтобы честно проверить, как модель работает на новых данных.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# === ПРИМЕР 3: Пайплайн машинного обучения ===

# Генерируем случайные тренировочные данные: X (признаки) и y (целевая переменная)
np.random.seed(42)
X_dummy = pd.DataFrame({
    'Признак_1': np.random.uniform(1, 10, 100),
    'Признак_2': np.random.uniform(20, 50, 100)
})
# Допустим, y зависит от признаков по формуле: y = 2*X1 + 0.5*X2 + шум
y_dummy = 2 * X_dummy['Признак_1'] + 0.5 * X_dummy['Признак_2'] + np.random.normal(0, 1, 100)

# 1. Разделение выборки (20% на тест, 80% на обучение)
X_train_ex, X_test_ex, y_train_ex, y_test_ex = train_test_split(
    X_dummy, y_dummy, test_size=0.2, random_state=42
)

print(f"Всего данных: {len(X_dummy)} строк")
print(f"Обучающая выборка (Train): {len(X_train_ex)} строк")
print(f"Тестовая выборка (Test): {len(X_test_ex)} строк\n")

# 2. Создание и обучение модели
model_ex = LinearRegression()
model_ex.fit(X_train_ex, y_train_ex) # Учится ТОЛЬКО на X_train и y_train!

# 3. Предсказание на новых данных
y_pred_ex = model_ex.predict(X_test_ex)

print("Первые 5 предсказаний модели:")
print(np.round(y_pred_ex[:5], 2))
print("Первые 5 истинных значений:")
print(np.round(y_test_ex.values[:5], 2))

### 3.4. Пример: Оценка качества модели (Метрики)
Чтобы понять, насколько хорошо модель предсказывает, используют метрики: MAE, MSE и $R^2$.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# === ПРИМЕР 4: Вычисление метрик ===

# Сравниваем истинные значения (y_test) и предсказанные (y_pred)
mae_ex = mean_absolute_error(y_test_ex, y_pred_ex)
mse_ex = mean_squared_error(y_test_ex, y_pred_ex)
r2_ex = r2_score(y_test_ex, y_pred_ex)

print("=== Оценка качества модели ===")
print(f"MAE (Средняя абсолютная ошибка): {mae_ex:.2f}")
print(f"MSE (Среднеквадратичная ошибка): {mse_ex:.2f}")
print(f"R² (Коэффициент детерминации)  : {r2_ex:.4f} (идеал = 1.0)")

# Вывод весов (влияние каждого признака)
print("\n=== Веса Линейной регрессии ===")
for col, coef in zip(X_dummy.columns, model_ex.coef_):
    print(f"{col}: {coef:.2f}")

### 3.5. Пример: Визуализация результатов регрессии (Matplotlib)
Лучший способ оценить регрессионную модель — построить график (Scatter plot), где по оси X отложены истинные значения, а по оси Y — предсказанные.

In [ ]:
import matplotlib.pyplot as plt

# === ПРИМЕР 5: График "Истина vs Предсказание" ===

plt.figure(figsize=(8, 6))

# Строим точки предсказаний (Истина по X, Предсказание по Y)
plt.scatter(y_test_ex, y_pred_ex, color='blue', alpha=0.6, label='Предсказания модели')

# Строим идеальную линию y = x (красный пунктир)
# Если модель идеальна, все синие точки лягут ровно на эту красную линию
min_val = y_test_ex.min()
max_val = y_test_ex.max()
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label='Идеальное совпадение')

# Оформление графика
plt.xlabel('Истинные значения (y_test)')
plt.ylabel('Предсказанные значения (y_pred)')
plt.title(f'Качество предсказаний (R² = {r2_ex:.2f})')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

---
## 4. Генерация данных (Блок преподавателя)
**Просто запустите эту ячейку.** Она генерирует сырой датасет с помощью физической модели, а затем имитирует аппаратные сбои (пропуски, текст, выбросы).

In [ ]:
# Физические константы
mu0 = 4.0 * np.pi * 1e-7
eps0 = 8.854187817e-12
Z0 = 376.73

def calc_shield_se(freq, table):
    """Метод матриц передачи."""
    if table.ndim == 1:
        table = table[np.newaxis, :]
    n_layers = table.shape[0]
    SE = np.zeros(len(freq))
    for i in range(len(freq)):
        w = 2.0 * np.pi * freq[i]
        A_total = np.eye(2, dtype=complex)
        for v in range(n_layers):
            mu_r, eps_r, sigma_v, t_v, mat_type = table[v]
            Ma = mu_r * mu0
            Ea = eps_r * eps0
            if int(mat_type) == 1:
                z = np.sqrt((1j * w * Ma) / (sigma_v + 1j * w * Ea))
                g = np.sqrt((1j * w * Ma) * (sigma_v + 1j * w * Ea))
            else:
                sig_comp = w * eps0 * np.imag(eps_r)
                z = (1 + 1j) * np.sqrt(w * Ma / (sig_comp + 1e-30))
                g = 1j * np.sqrt(w * Ma * (sig_comp + 1j * w * eps0 * np.real(eps_r)))
            A_layer = np.array([
                [np.cosh(g * t_v), z * np.sinh(g * t_v)],
                [np.sinh(g * t_v) / z, np.cosh(g * t_v)]
            ], dtype=complex)
            A_total = A_total @ A_layer
        T = 2 * Z0 / (A_total[1, 0] * Z0**2 + A_total[1, 1] * Z0 + A_total[0, 0] * Z0 + A_total[0, 1])
        SE[i] = 20.0 * np.log10(np.abs(1.0 / T))
    return SE

# === ГЕНЕРАЦИЯ ДАННЫХ ===
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
N = 3000

# Варьируем 3 параметра: Частота, mu_r, Проводимость
freq_hz = 10 ** np.random.uniform(6, 9, N) # 1 МГц - 1 ГГц
freq_ghz = freq_hz / 1e9
mu_r = np.random.uniform(1, 100, N)
sigma_msm = np.random.uniform(1, 100, N)

# Константы (металл, 1 мкм)
t = np.full(N, 1e-6)
eps_r = np.ones(N)

print("Рассчитываем аналитическую модель (SE)...")
se_values = np.zeros(N)
for idx in range(N):
    # Подаем СИ (См/м) в функцию
    layer = np.array([mu_r[idx], eps_r[idx], sigma_msm[idx] * 1e6, t[idx], 1])
    se_values[idx] = calc_shield_se(np.array([freq_hz[idx]]), layer)[0]

df_ideal = pd.DataFrame({
    'freq_GHz': freq_ghz,
    'mu_r': mu_r,
    'sigma_MSm': sigma_msm,
    'SE': se_values
})

# === ИСКУССТВЕННОЕ ЗАГРЯЗНЕНИЕ ===
df_dirty = df_ideal.copy()
df_dirty['sigma_MSm'] = df_dirty['sigma_MSm'].astype(object)
rng = np.random.default_rng(42)

# 1. Пропуски (NaN) в частоте
df_dirty.loc[rng.choice(N, 60, replace=False), 'freq_GHz'] = np.nan
# 2. Текстовый мусор в проводимости
df_dirty.loc[rng.choice(N, 50, replace=False), 'sigma_MSm'] = rng.choice(['ошибка_чтения', 'N/A'])
# 3. Выбросы (Z-score) в mu_r: невозможные значения для наших данных
outlier_mu = rng.choice(N, 40, replace=False)
df_dirty.loc[outlier_mu, 'mu_r'] = rng.uniform(8000, 15000, 40)
# 4. Выбросы (IQR) в SE: сбой итогового расчёта
outlier_se = rng.choice(N, 30, replace=False)
df_dirty.loc[outlier_se, 'SE'] = df_dirty.loc[outlier_se, 'SE'] + 80

df = df_dirty.sample(frac=1, random_state=42).reset_index(drop=True)
print("Сырой датасет успешно сгенерирован в переменную `df`!")

---
## 📝 Практика 1: Очистка типов и пустых значений (Pandas)
В столбце `sigma_MSm` есть текстовые ошибки, из-за которых Pandas сделал весь столбец текстовым (`object`). Машинное обучение работает только с числами.

In [ ]:
# [Выведите количество пропусков (NaN) в каждом столбце df]
nulls_before =
print("Пропуски до очистки:\n", nulls_before)

# [Преобразуйте столбец 'sigma_MSm' в числовой формат (float). Весь нечитаемый текст должен стать NaN]
df['sigma_MSm'] =

# [Удалите все строки с пропусками (NaN) из датафрейма df и сохраните в df_dropped]
df_dropped =

# [Выведите размер датафрейма после удаления пустых строк]
print(f"Размер после удаления NaN: {df_dropped.shape}")

---
## 📝 Практика 2: Удаление выбросов (IQR и Z-score)
В данных остались физически невозможные выбросы: как в магнитной проницаемости (`mu_r`), так и в итоговом `SE`.

In [ ]:
# --- 1. Очистка 'SE' методом IQR ---

# [Рассчитайте первый (Q1, 0.25) и третий (Q3, 0.75) квартили для столбца 'SE']
q1 =
q3 =

# [Рассчитайте межквартильный размах IQR]
iqr = q3 - q1

# [Отфильтруйте df_dropped: оставьте строки, где 'SE' находится строго в границах [Q1 - 1.5*IQR; Q3 + 1.5*IQR]. Сохраните в df_iqr]
lower_bound =
upper_bound =
df_iqr =


# --- 2. Очистка 'mu_r' методом Z-score ---

# [Рассчитайте модуль Z-оценки для столбца 'mu_r' в датафрейме df_iqr]
z_scores_mu =

# [Отфильтруйте df_iqr: оставьте только те строки, где Z-оценка < 3. Сохраните результат в итоговый датасет df_clean]
df_clean =

print(f"Размер идеально чистого датасета: {df_clean.shape}")

---
## 📝 Практика 3: Обучение моделей машинного обучения
Теперь мы попробуем аппроксимировать аналитическую модель матриц передачи двумя способами:
1. **Линейная регрессия** (пытается провести прямую плоскость сквозь данные).
2. **Случайный лес (Random Forest)** (ансамбль деревьев решений, способный описывать сложнейшие нелинейные изгибы).

In [ ]:
import matplotlib.pyplot as plt

# [Выделите матрицу признаков X ('freq_GHz', 'mu_r', 'sigma_MSm') и целевую переменную y ('SE') из df_clean]
X =
y =

# [Разделите выборку на обучающую (Train) и тестовую (Test). Тест = 20%, random_state=42]
X_train, X_test, y_train, y_test =

# === МОДЕЛЬ 1: ЛИНЕЙНАЯ РЕГРЕССИЯ ===
# [Создайте и обучите LinearRegression на обучающих данных]
lr_model =
lr_model.fit(X_train, y_train)

# [Сделайте предсказание y_pred_lr на тестовых данных]
y_pred_lr =

# [Рассчитайте MAE, MSE и R2 для Линейной регрессии]
mae_lr =
mse_lr =
r2_lr =


# === МОДЕЛЬ 2: СЛУЧАЙНЫЙ ЛЕС (Random Forest) ===
# [Создайте и обучите RandomForestRegressor (укажите random_state=42) на обучающих данных]
rf_model =
rf_model.fit(X_train, y_train)

# [Сделайте предсказание y_pred_rf на тестовых данных]
y_pred_rf =

# [Рассчитайте MAE, MSE и R2 для Случайного леса]
mae_rf =
mse_rf =
r2_rf =

# --- Вывод результатов ---
print("=== СРАВНЕНИЕ КАЧЕСТВА МОДЕЛЕЙ ===")
print(f"Линейная Регрессия: MAE = {mae_lr:.2f} дБ, MSE = {mse_lr:.2f}, R² = {r2_lr:.4f}")
print(f"Случайный Лес     : MAE = {mae_rf:.2f} дБ, MSE = {mse_rf:.2f}, R² = {r2_rf:.4f}")

---
## 📝 Практика 4: Визуализация предсказаний
Построим графики сравнения истинных значений SE и предсказанных. Идеальные предсказания ложатся строго на красную пунктирную линию.

In [ ]:
# [Создайте фигуру с двумя графиками (subplots 1x2) размером 14x6]


# --- График 1: Линейная регрессия ---
# [Постройте scatter plot истинных (y_test) vs предсказанных (y_pred_lr) значений. Цвет: синий, alpha=0.5]

# --- График 2: Случайный лес ---
# [Постройте scatter plot истинных (y_test) vs предсказанных (y_pred_rf) значений. Цвет: зелёный, alpha=0.5]


plt.tight_layout()
plt.show()

---
## 🌟 ДОПОЛНИТЕЛЬНОЕ ЗАДАНИЕ (на оценку)

Мы выяснили, что Random Forest легко справляется с задачей. Теперь усложним симуляцию: добавим в генерацию вариативную **толщину экрана** $t$.

Теперь у нас **4 варьируемых параметра**:
1. `freq_GHz` (от 1 МГц до 1 ГГц)
2. `mu_r` (от 1 до 100)
3. `sigma_MSm` (от 1 до 100 МСм/м)
4. `thickness_um` (Толщина: от 1 до 50 микрометров)

**Ваша задача**:
1. Запустить генератор новых данных.
2. Выделить признаки $X$ (все 4 штуки) и $y$ ($SE$).
3. Обучить **RandomForestRegressor** на новых данных.
4. Вывести метрику $R^2$.
5. С помощью встроенного атрибута `model.feature_importances_` определить, **какой из физических параметров вносит наибольший вклад** в эффективность экранирования в этом датасете.

In [ ]:
# --- Генерация сложных 4D данных (Блок преподавателя) ---
np.random.seed(99)
N_complex = 2000
freq_comp = 10 ** np.random.uniform(6, 9, N_complex)
mu_comp = np.random.uniform(1, 100, N_complex)
sigma_comp = np.random.uniform(1, 100, N_complex)
thick_comp = np.random.uniform(1e-6, 50e-6, N_complex) # от 1 мкм до 50 мкм
se_comp = np.zeros(N_complex)

print("Генерация сложных данных с толщиной. Подождите...")
for i in range(N_complex):
    layer = np.array([mu_comp[i], 1.0, sigma_comp[i] * 1e6, thick_comp[i], 1])
    se_comp[i] = calc_shield_se(np.array([freq_comp[i]]), layer)[0]

df_complex = pd.DataFrame({
    'freq_GHz': freq_comp / 1e9,
    'mu_r': mu_comp,
    'sigma_MSm': sigma_comp,
    'thickness_um': thick_comp * 1e6,
    'SE': se_comp
})
print("Готово!")

# === ВАШ КОД НИЖЕ ===

# [Выделите X (4 признака) и y (SE) из df_complex]
X_comp = df_complex[['freq_GHz', 'mu_r', 'sigma_MSm', 'thickness_um']]
y_comp = df_complex['SE']

# [Разделите данные на Train и Test (test_size=0.2)]
X_tr, X_te, y_tr, y_te = train_test_split(X_comp, y_comp, test_size=0.2, random_state=42)

# [Обучите RandomForestRegressor]
rf_complex = RandomForestRegressor(random_state=42)
rf_complex.fit(X_tr, y_tr)

# [Вычислите и выведите R2 на тестовой выборке]
pred_comp = rf_complex.predict(X_te)
r2_comp = r2_score(y_te, pred_comp)
print(f"R² сложной модели (4 признака): {r2_comp:.4f}")

# [Извлеките важность признаков с помощью rf_complex.feature_importances_]
importances = rf_complex.feature_importances_
feature_names = ['freq_GHz', 'mu_r', 'sigma_MSm', 'thickness_um']

# [Выведите важность каждого признака в процентах]
print("\n=== ВАЖНОСТЬ ПРИЗНАКОВ (Влияние на SE) ===")
for name, imp in zip(feature_names, importances):
    print(f"{name:15s}: {imp*100:.1f}%")

